In [8]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. 데이터 불러오기
df = pd.read_csv('https://raw.githubusercontent.com/lgh8615-cmd/maturity-of-plant/refs/heads/main/plant_maturity_high_accuracy_150.csv')
# 2. DTM 값을 기준으로 성숙 단계 나누기
maturity_list = []

for dtm in df['DTM']:
    if dtm < 70:
        maturity_list.append('전기')
    elif dtm < 95:
        maturity_list.append('중기')
    else:
        maturity_list.append('후기')

df['maturity_class'] = maturity_list


# 3. 입력 데이터와 타깃 데이터 정하기
feature = df[['Band_All6', 'MCARI', 'NDVI']]
target = df['maturity_class']

# 4. 훈련 세트와 테스트 세트 나누기
train_input, test_input, train_target, test_target = train_test_split(
    feature, target, stratify=target, random_state=42)

# 5. 다항 특성 만들기
poly = PolynomialFeatures(include_bias=False)
train_poly = poly.fit_transform(train_input)
test_poly = poly.transform(test_input)

print("변환 전 특성 개수:", train_input.shape[1])
print("변환 후 특성 개수:", train_poly.shape[1])
print()

# 6. 정규화
ss = StandardScaler()
train_scaled = ss.fit_transform(train_poly)
test_scaled = ss.transform(test_poly)

# 7. K-최근접 이웃 분류 모델 훈련
kn = KNeighborsClassifier(n_neighbors=3)
kn.fit(train_scaled, train_target)

# 8. 정확도 확인
train_score = kn.score(train_scaled, train_target)
test_score = kn.score(test_scaled, test_target)

print("훈련 세트 정확도:", train_score)
print("테스트 세트 정확도:", test_score)


변환 전 특성 개수: 3
변환 후 특성 개수: 9

훈련 세트 정확도: 0.9017857142857143
테스트 세트 정확도: 0.7631578947368421


훈련세트 정확도는 높지만, 그에 비해 테스트 정확도는 낮다.
따라서 해당 데이터로 학습시키면 과대적합 문제가 발생한다.

해결하기 위해, 훈련 세트와 테스트 세트의 정확도 모두 높은 편이고, 두 점수가 비슷한 적절한 이웃의 개수를 찾아야 한다.

In [21]:
# 9. 과대적합 문제 해결하기

# 이웃 수를 늘려가며 훈련 세트와 테스트 세트의 점수가 비슷한 이웃 찾아내기
good_n_neightbors=3 # 초기값
subtrait_train_test = 1 # 훈련 세트와 테스트 세트의 점수 차이 초기값

for n in range(3, 30):

  # 이웃 수 바꿔 훈련시키기
  kn = KNeighborsClassifier(n_neighbors=n)
  kn.fit(train_scaled, train_target)

  # 훈련세트와 테스트세트의 정확도 차이 리스트에 저장하기
  train_score = kn.score(train_scaled, train_target)
  test_score = kn.score(test_scaled, test_target)
  if subtrait_train_test > train_score - test_score:
    subtrait_train_test = train_score - test_score
    good_n_neightbors = n

# 정확도 차이가 가장 작은 이웃 수 출력
print("모델에 가장 적합한 이웃 수:", good_n_neightbors)

모델에 가장 적합한 이웃 수: 17


In [22]:
# 10. 가장 적합한 이웃 수로 다시 훈련시키기
kn = KNeighborsClassifier(n_neighbors=good_n_neightbors)
kn.fit(train_scaled, train_target)

# 11. 다시 정확도 확인
train_score = kn.score(train_scaled, train_target)
test_score = kn.score(test_scaled, test_target)

print("훈련 세트 정확도:", train_score)
print("테스트 세트 정확도:", test_score)


훈련 세트 정확도: 0.8928571428571429
테스트 세트 정확도: 0.868421052631579


1. 훈련세트와 테스트 세트 모두 정확도가 높은 편이다.
2. 훈련세트와 테스트 세트 정확도 차이가 가장 작다.

따라서 현재 상태는 적절하게 모델을 훈련시키기에 적합하다.

In [24]:
# 12. 훈련시킨 모델 가지고 예측해보기
new_data = [[0.85, 0.65, 0.90]]

# 다항 변환
new_poly = poly.transform(new_data)

# 정규화(훈련세트에 맞춰서!!)
new_scaled = ss.transform(new_poly)

# 예측
pred = kn.predict(new_scaled)

print(pred)

['전기']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(
